In [2]:
import torch
import sqlparse
import random
from datasets import load_dataset, Dataset
from transformers import (
    T5Tokenizer,
    T5ForConditionalGeneration,
    Trainer,
    TrainingArguments,
    DataCollatorForSeq2Seq
)
from tqdm import tqdm

device = "cuda" if torch.cuda.is_available() else "cpu"

# ==========================================================
# 1. LOAD DATASET
# ==========================================================
print("Loading dataset a3467895898/bird_text2sql...")
dataset = load_dataset("a3467895898/bird_text2sql")["train"]

# Split train / test
TEST_SIZE = 200
TRAIN_SIZE = 5000

train_data = dataset.select(range(TRAIN_SIZE))
test_data   = dataset.select(range(TEST_SIZE))

print(f"Train samples: {len(train_data)}")
print(f"Test samples  : {len(test_data)}")

# ==========================================================
# 2. SQL NORMALIZATION
# ==========================================================
def normalize_sql(sql: str):
    return sqlparse.format(
        sql.strip().rstrip(";"),
        keyword_case="upper",
        strip_comments=True,
        reindent=False
    )

# ==========================================================
# 3. INPUT PARSING (SCHEMA + QUESTION)
# ==========================================================
def extract_schema_and_question(input_full_str: str):
    schema_prefix = "[INST] Here is a database schema:\n"
    question_prefix = "\n\nPlease write me a SQL statement that answers the following question: "
    inst_suffix = " [/INST]"

    schema_start = input_full_str.find(schema_prefix) + len(schema_prefix)
    question_start = input_full_str.find(question_prefix)

    schema = input_full_str[schema_start:question_start].strip()
    question = input_full_str[
        question_start + len(question_prefix):
    ].replace(inst_suffix, "").strip()

    return schema, question

# ==========================================================
# 4. PREPROCESS DATA (SCHEMA-AWARE)
# ==========================================================
def preprocess(example):
    schema, question = extract_schema_and_question(example["input"])

    input_text = (
        "Database schema:\n"
        + schema
        + "\n\nQuestion:\n"
        + question
    )

    return {
        "input_text": input_text,
        "target_text": normalize_sql(example["output"])
    }

train_processed = [preprocess(x) for x in train_data]
test_processed   = [preprocess(x) for x in test_data]

train_ds = Dataset.from_list(train_processed)
test_ds   = Dataset.from_list(test_processed)

# ==========================================================
# 5. MODEL & TOKENIZER
# ==========================================================
MODEL_NAME = "t5-small"

tokenizer = T5Tokenizer.from_pretrained(MODEL_NAME)
model = T5ForConditionalGeneration.from_pretrained(MODEL_NAME).to(device)

MAX_INPUT_LEN  = 512
MAX_OUTPUT_LEN = 256

def tokenize(batch):
    inputs = tokenizer(
        batch["input_text"],
        max_length=MAX_INPUT_LEN,
        truncation=True,
        padding="max_length"
    )
    labels = tokenizer(
        batch["target_text"],
        max_length=MAX_OUTPUT_LEN,
        truncation=True,
        padding="max_length"
    )
    inputs["labels"] = labels["input_ids"]
    return inputs

train_ds = train_ds.map(tokenize, batched=True, remove_columns=train_ds.column_names)
test_ds  = test_ds.map(tokenize, batched=True, remove_columns=test_ds.column_names)

# ==========================================================
# 6. TRAINING CONFIG (FINE-TUNING)
# ==========================================================
training_args = TrainingArguments(
    output_dir="./text2sql_model",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=3e-4,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,
    num_train_epochs=10,
    fp16=torch.cuda.is_available(),
    logging_steps=100,
    save_total_limit=2,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=test_ds,
    processing_class=tokenizer, # Changed from tokenizer=tokenizer
    data_collator=DataCollatorForSeq2Seq(tokenizer, model)
)

# ==========================================================
# 7. FINE-TUNE MODEL
# ==========================================================
trainer.train()

# ==========================================================
# 8. SAVE MODEL
# ==========================================================
model.save_pretrained("./final_text2sql_model")
tokenizer.save_pretrained("./final_text2sql_model")
print("✅ Model saved to ./final_text2sql_model")

# ==========================================================
# 9. TEXT → SQL GENERATION
# ==========================================================
def generate_sql(question, schema):
    prompt = (
        "Database schema:\n"
        + schema
        + "\n\nQuestion:\n"
        + question
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_INPUT_LEN
    ).to(device)

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_length=MAX_OUTPUT_LEN,
            num_beams=4
        )

    return tokenizer.decode(output[0], skip_special_tokens=True)

# ==========================================================
# 10. ACCURACY METRICS
# ==========================================================
def exact_match(pred, gold):
    return normalize_sql(pred) == normalize_sql(gold)

def token_accuracy(pred, gold):
    p = set(normalize_sql(pred).split())
    g = set(normalize_sql(gold).split())
    return len(p & g) / max(1, len(g))

# ==========================================================
# 11. RANDOM EVALUATION
# ==========================================================
N = 20
indices = random.sample(range(len(test_data)), N)

exact_correct = 0
token_scores = []

print("\n🔍 Evaluating random samples...\n")

for idx in indices:
    ex = test_data[idx]
    schema, question = extract_schema_and_question(ex["input"])
    gold_sql = ex["output"]

    pred_sql = generate_sql(question, schema)

    print("\n" + "=" * 80)
    print("Question:")
    print(question)
    print("\nPredicted SQL:")
    print(pred_sql)
    print("\nGold SQL:")
    print(gold_sql)

    if exact_match(pred_sql, gold_sql):
        exact_correct += 1

    token_scores.append(token_accuracy(pred_sql, gold_sql))

print("\n========== RESULTS ==========")
print("Exact Match Accuracy :", exact_correct / N)
print("Token Accuracy       :", sum(token_scores) / len(token_scores))
print("============================")

Loading dataset a3467895898/bird_text2sql...
Train samples: 5000
Test samples  : 200


Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

Epoch,Training Loss,Validation Loss
1,0.240100,0.137444
2,0.173600,0.101211
3,0.152100,0.083454
4,0.131800,0.073780
5,0.119700,0.065092
6,0.108700,0.060549
7,0.100700,0.057117
8,0.096600,0.054545
9,0.091000,0.053500
10,0.090100,0.052428


✅ Model saved to ./final_text2sql_model

🔍 Evaluating random samples...


Question:
What is the average number of number of movies added to the lists of user 8516503? Indicate how many movies did he/she give a rating score of 5.

Predicted SQL:
SELECT T3.list_title FROM ratings AS T1 INNER JOIN lists_users AS T2 ON T1.user_id = T2.user_id INNER JOIN LISTS AS T3 ON T2.list_id = T3.list_id INNER JOIN movies AS T4 ON T3.movie_id = T4.movie_id WHERE T1.rating_timestamp_utc LIKE 'T1.rating_score'

Gold SQL:
SELECT AVG(T3.list_movie_number) , SUM(CASE WHEN T1.rating_score = 5 THEN 1 ELSE 0 END) FROM ratings AS T1 INNER JOIN lists_users AS T2 ON T1.user_id = T2.user_id INNER JOIN lists AS T3 ON T2.user_id = T3.user_id WHERE T1.user_id = 8516503;

Question:
How many movies were added to the list with the most number of movies? Indicate whether the user was a paying subscriber or not when he created the list.

Predicted SQL:
SELECT T2.list_movie_number, T2.user_has_payment_method FROM LISTS AS 